# Épidémies sur réseau : le seuil que les degrés décident

[↑ Dynamiques](README.md) · [↑ Bio & Médecine](../README.md)

Un SIR à population homogène se résume à un nombre, $R_0$ : au-dessus de un, l'épidémie monte ; en dessous, elle s'éteint. Dès que les contacts ont une structure, ce nombre cesse de suffire. Deux populations de même degré moyen peuvent porter l'une une flambée, l'autre rien, parce que ce qui décide n'est pas le nombre moyen de contacts mais la façon dont ces contacts se distribuent et se referment.

Ce carnet construit un réseau de contacts à la main, y simule une épidémie par une file d'événements à temps continu, puis la juge par une voie qui ne partage ni code, ni structures de données, ni aléa : la percolation statique, lue par un union-find. Les deux voies doivent s'accorder là où la théorie le dit — et le carnet mesure précisément **où elles ne s'accordent pas** : sur un SIR à horloge de guérison partagée, les transmissions d'un même sujet ne sont pas indépendantes, et la taille finale reste sous la prédiction de percolation. Le contre-témoin, enfin, construit un graphe où la formule de seuil elle-même se trompe.

## Ce que ce carnet établit, et ce qu'il n'établit pas

| Section | Question posée | Ce qui la tranche |
|---|---|---|
| 1. Modèle | Que devient un SIR quand les contacts forment un graphe ? | Le graphe, ses degrés, et leurs deux premiers moments |
| 2. Calcul | Combien de personnes l'épidémie atteint-elle ? | Une file d'événements : guérisons et tentatives tirées à dates exponentielles |
| 3. Contrôle indépendant | Ce nombre est-il une propriété du graphe ou un artefact du simulateur ? | La percolation statique : chaque arête gardée avec la probabilité $T$, puis union-find |
| 4. Contre-témoin | La formule de seuil sait-elle se tromper ? | Un anneau triangulé, de degré 4, où elle annonce une flambée que la mesure refuse |
| 5. Statut | Que peut-on affirmer, et à quel titre ? | La distinction mesuré / cité / ouvert, en fin de carnet |

Le carnet ne dit rien des populations réelles : les réseaux sont simulés à paramètres connus, aucune donnée de contact n'est utilisée. Il ne dit rien non plus de l'inférence — retrouver $T$ et la structure à partir d'une épidémie observée est un autre problème, laissé ouvert.

## Outils et conventions

Le carnet n'utilise que la bibliothèque standard (`heapq`) et `numpy`. Aucune bibliothèque de graphes n'est employée, et c'est délibéré : le contrôle de la section 3 doit être indépendant du calcul de la section 2, or deux appels à la même bibliothèque partageraient la même implémentation, donc les mêmes erreurs. Les deux voies sont écrites séparément et ne se rencontrent qu'à la comparaison.

Deux conventions de lecture :

- Une **arête** relie deux sujets qui se rencontrent ; $T$ est la probabilité qu'un contact donné transmette.
- Un **amas** est un ensemble de sujets reliés par des transmissions effectives. La **taille finale** d'une épidémie partie d'un cas unique est la taille de l'amas de ce cas.

Une distinction statistique commande toute la section 3. La *fraction de l'amas géant* est la taille du plus grand amas rapportée à la population. La *taille attendue depuis une graine uniforme* vaut $\sum_c (|c|/n)^2$ sur les amas $c$ : c'est la seule quantité comparable à la moyenne d'une simulation partie d'un seul cas tiré au hasard. Les deux diffèrent, et confondre l'une avec l'autre suffit à fabriquer un désaccord qui n'existe pas.

Les tirages sont reproductibles : chaque générateur part d'une graine fixe.

In [1]:
import heapq

import numpy as np

# Graine fixe : le carnet doit rejouer les memes nombres d'une execution a l'autre.
GRAINE = 20261127

## 1. Modèle — un SIR dont les contacts ont une structure

On reprend le SIR de [Dynamiques-01](Dynamiques-01-Populations-et-Epidemies-Python.ipynb), avec une différence qui change tout : il n'y a plus de population indifférenciée. Chaque sujet a une liste finie de contacts et ne peut infecter que ceux-là.

Trois hypothèses, explicites parce que ce sont elles qui décideront :

1. **Le graphe est fixe.** Les contacts ne se créent ni ne se défont pendant l'épidémie — raisonnable sur une durée courte devant celle des relations sociales.
2. **Une horloge de guérison par sujet.** À l'infection, un sujet tire une durée infectieuse $\tau \sim \mathrm{Exp}(\gamma)$ ; pendant $\tau$, chaque contact est une tentative de transmission tirée à $\mathrm{Exp}(\beta)$. La probabilité qu'une tentative aboutisse avant la guérison vaut alors $T = \beta/(\beta+\gamma)$. Le carnet fixe $\gamma = 1$, donc $\beta = T/(1-T)$.
3. **Immunité définitive.** Un sujet guéri n'est plus infectieux et ne redevient jamais sensible.

L'hypothèse 2 mérite qu'on s'y arrête, car elle porte une conséquence mesurable. Les tentatives d'un même sujet partagent **une seule** horloge de guérison : si ce sujet guérit vite, toutes ses transmissions échouent ensemble. Les arêtes d'un même sommet ne sont donc pas indépendantes, même si chacune transmet avec la probabilité $T$ en moyenne. La section 3 mesurera ce que cela change.

Le réseau de référence est un graphe aléatoire à degré moyen visé, construit par tirage d'arêtes avec rejet des doublons. Sa distribution de degrés est à peu près de Poisson — le cas le plus favorable aux formules de mélange homogène, celles qu'on cherche justement à éprouver.

In [2]:
def construire_reseau(n, degre_moyen, generateur):
    """Graphe aleatoire a degre moyen vise, construit a la main (listes d'adjacence).

    Chaque arete est tiree uniformement parmi les paires de sommets distincts ;
    doublons et boucles sont rejetes, donc le graphe est simple."""
    m = int(round(n * degre_moyen / 2))          # nombre d'aretes vise
    adj = [set() for _ in range(n)]
    aretes = 0
    while aretes < m:
        i, j = int(generateur.integers(0, n)), int(generateur.integers(0, n))
        if i == j or j in adj[i]:
            continue                              # ni boucle sur soi, ni arete multiple
        adj[i].add(j)
        adj[j].add(i)
        aretes += 1
    return [sorted(a) for a in adj]


def moment_degres(adj):
    """Degre moyen et moment d'ordre deux : les deux quantites qui commandent le seuil."""
    degres = np.array([len(a) for a in adj], dtype=float)
    return degres.mean(), float((degres ** 2).mean())


n_reseau = 3000
generateur = np.random.default_rng(GRAINE)
reseau = construire_reseau(n_reseau, 6.0, generateur)
k_moyen, k2_moyen = moment_degres(reseau)

# Seuil d'un reseau a degres prescrits : T_c = <k> / (<k^2> - <k>).
# Il se lit sur les seuls moments, sans simuler une seule infection.
seuil_reseau = k_moyen / (k2_moyen - k_moyen)
# Prediction du melange homogene, qui ne connait que le degre moyen.
seuil_homogene = 1.0 / k_moyen

print(f"sommets={n_reseau}  aretes={sum(len(a) for a in reseau) // 2}")
print(f"<k>={k_moyen:.3f}  <k^2>={k2_moyen:.3f}")
print(f"seuil du reseau    T_c = <k>/(<k^2>-<k>) = {seuil_reseau:.4f}")
print(f"seuil homogene     T_c = 1/<k>           = {seuil_homogene:.4f}")

sommets=3000  aretes=9000
<k>=6.000  <k^2>=42.124
seuil du reseau    T_c = <k>/(<k^2>-<k>) = 0.1661
seuil homogene     T_c = 1/<k>           = 0.1667


### Lecture du résultat

Les deux formules donnent ici presque le même seuil, et ce n'est pas un hasard : sur des degrés de Poisson, $\langle k^2\rangle = \langle k\rangle^2+\langle k\rangle$, donc la formule de réseau se réduit à $1/\langle k\rangle$ exactement. L'écart mesuré, `0,0006`, n'est que celui du tirage fini : la distribution réalisée n'est pas exactement de Poisson.

Cette coïncidence est un piège pédagogique — elle laisse croire que le degré moyen suffit — et une commodité : elle fixe l'ordre de grandeur. Avec six contacts en moyenne, une transmission par contact au-delà de `0,1661` suffit à faire flamber le réseau. La section 4 montrera un graphe où les deux formules, elles, divergent franchement.

## 2. Calcul — la simulation par file d'événements

Le SIR sur réseau n'est pas un système d'équations différentielles : c'est une suite d'événements discrets dont les dates sont aléatoires. On le simule par une file de priorité où chaque événement porte sa date absolue.

À l'infection d'un sujet, deux familles d'événements sont programmées : sa guérison, à une date tirée exponentiellement de taux $\gamma$, et une tentative vers chacun de ses contacts, à une date tirée exponentiellement de taux $\beta$. Une tentative n'aboutit que si la source est encore infectieuse à sa date et si la cible est encore saine ; sinon elle est ignorée, ce qui est exactement la sémantique d'un SIR — une occasion manquée ne se rattrape pas.

La fonction prend un argument `modele`. Avec `horloge` (le modèle physique de la section 1), toutes les tentatives d'un sujet partagent sa guérison. Avec `par_arete`, chaque tentative tire au contraire son propre sort, indépendamment de toute horloge : c'est le modèle de percolation, et il servira de pont vers la section 3.

In [3]:
def epidemie(adj, T, graine, generateur, modele="horloge", gamma=1.0):
    """SIR sur reseau par file d'evenements a temps continu.

    modele="horloge" : chaque sujet tire une duree infectieuse ~ Exp(gamma) et, pour
    chaque voisin, une date de tentative ~ Exp(beta) avec beta = T/(1-T). Une tentative
    aboutit si elle tombe avant la guerison de la source.
    modele="par_arete" : chaque tentative reussit avec la probabilite T, sans horloge.
    Retourne la fraction finale de la population atteinte (infectee ou guerrie).
    """
    n = len(adj)
    beta = T / (1.0 - T)
    etat = [0] * n                     # 0 sain, 1 infectieux, 2 gueri
    tas = []

    def infecter(sujet, instant):
        etat[sujet] = 1
        if modele == "horloge":
            heapq.heappush(tas, (instant + generateur.exponential(1.0 / gamma), 0, sujet, -1))
        for voisin in adj[sujet]:
            if modele == "horloge":
                date = instant + generateur.exponential(1.0 / beta)
            else:
                date = instant          # pas d'horloge : le sort est tire a l'infection
            heapq.heappush(tas, (date, 1, sujet, voisin))

    infecter(graine, 0.0)
    while tas:
        date, genre, sujet, voisin = heapq.heappop(tas)
        if genre == 0:
            etat[sujet] = 2                        # guerison
            continue
        if modele == "par_arete" and generateur.random() >= T:
            continue                               # tentative perdue
        if etat[sujet] == 1 and etat[voisin] == 0:
            infecter(voisin, date)                 # transmission reussie
    return sum(1 for e in etat if e != 0) / n

## 3. Contrôle indépendant — la percolation statique, sans temps ni parcours

La simulation précédente a un défaut méthodologique : elle produit le résultat que son code veut bien produire. Si la file d'événements avait un bogue — un événement ignoré à tort, une date comparée dans le mauvais sens — elle rendrait un nombre plausible et faux. Il faut un juge qui ne partage rien avec elle.

Ce juge est connu et exact pour un modèle voisin. Si chaque arête transmet **indépendamment** avec la probabilité $T$, l'ensemble final des sujets atteints a la loi d'un amas de percolation : on garde chaque arête avec la probabilité $T$, puis on prend la composante du cas initial. Aucune notion de temps, aucune guérison, aucun parcours d'épidémie — un tirage d'arêtes et un comptage de composantes.

L'implémentation n'utilise donc ni file de priorité, ni exponentielle : un tirage uniforme par arête et un union-find. C'est ce qui rend la comparaison concluante, à condition de comparer **la même statistique** des deux côtés : la taille attendue depuis une graine uniforme, $\sum_c (|c|/n)^2$, et non la fraction de l'amas géant.

In [4]:
def statistiques_percolation(adj, probabilite, generateur):
    """Percolation statique : chaque arete survit avec la probabilite donnee, puis union-find.

    Retourne (fraction du plus grand amas, taille attendue depuis une graine uniforme,
    nombre d'amas). Aucune notion de temps ni d'epidemie : que des composantes.
    """
    n = len(adj)
    parent = list(range(n))

    def racine(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]      # compression de chemin
            x = parent[x]
        return x

    for i in range(n):
        for j in adj[i]:
            if j > i and generateur.random() < probabilite:
                a, b = racine(i), racine(j)
                if a != b:
                    parent[a] = b
    tailles = {}
    for i in range(n):
        r = racine(i)
        tailles[r] = tailles.get(r, 0) + 1
    liste = np.array(list(tailles.values()), dtype=float)
    # Esperance de la taille atteinte depuis une graine tiree uniformement : somme des carres.
    return liste.max() / n, float(((liste / n) ** 2).sum()), len(liste)

In [5]:
transmissibilites = np.array([0.10, 0.13, 0.16, 0.18, 0.22, 0.30])
n_tirages = 400
n_blocs = 10
taille_bloc = n_tirages // n_blocs

# Un generateur par voie : les trois colonnes ne partagent aucun flux d'alea, et changer
# le nombre de tirages d'une colonne ne deplace pas les nombres des deux autres.
generateur_perc = np.random.default_rng(GRAINE + 101)
generateur_horloge = np.random.default_rng(GRAINE + 102)
generateur_arete = np.random.default_rng(GRAINE + 103)

print("   T   percolation   SIR horloge   SIR par arete   ecart(perc,arete)   ecart(perc,horloge)")
ecarts_arete, ecarts_horloge = [], []
mesures = []
for T in transmissibilites:
    # Voie 1 : percolation statique, taille attendue depuis une graine uniforme.
    perc = [statistiques_percolation(reseau, T, generateur_perc)[1] for _ in range(n_tirages)]
    # Voies 2 et 3 : la meme file d'evenements, deux modeles de transmission.
    horl = [epidemie(reseau, T, int(generateur_horloge.integers(0, n_reseau)), generateur_horloge,
                     "horloge") for _ in range(n_tirages)]
    aret = [epidemie(reseau, T, int(generateur_arete.integers(0, n_reseau)), generateur_arete,
                     "par_arete") for _ in range(n_tirages)]
    mp, mh, ma = float(np.mean(perc)), float(np.mean(horl)), float(np.mean(aret))
    ecarts_arete.append(abs(mp - ma))
    ecarts_horloge.append(abs(mp - mh))
    mesures.append((T, perc, horl))
    print(f"{T:5.2f}   {mp:11.3f}   {mh:11.3f}   {ma:13.3f}   "
          f"{abs(mp - ma):17.3f}   {abs(mp - mh):18.3f}")

print(f"\necart maximal percolation / par arete : {max(ecarts_arete):.4f}")
print(f"ecart maximal percolation / horloge   : {max(ecarts_horloge):.4f}")

# Une moyenne sur n_tirages ne dit rien de sa propre dispersion. On decoupe les tirages en
# blocs pour voir de combien l'ecart bouge d'un bloc a l'autre : c'est la question de
# l'exercice 1, et elle fixe la precision du chiffre qu'on vient d'imprimer.
pire = int(np.argmax(ecarts_horloge))
T_pire, perc_pire, horl_pire = mesures[pire]
blocs_perc = [float(np.mean(perc_pire[i * taille_bloc:(i + 1) * taille_bloc])) for i in range(n_blocs)]
blocs_horl = [float(np.mean(horl_pire[i * taille_bloc:(i + 1) * taille_bloc])) for i in range(n_blocs)]
ecarts_bloc = np.abs(np.array(blocs_perc) - np.array(blocs_horl))

print(f"\nT = {T_pire:.2f} : ecart mesure sur {n_blocs} blocs de {taille_bloc} tirages")
print(f"  percolation par bloc : {min(blocs_perc):.3f} a {max(blocs_perc):.3f}   "
      f"(ecart-type {np.std(blocs_perc, ddof=1):.4f})")
print(f"  SIR horloge par bloc : {min(blocs_horl):.3f} a {max(blocs_horl):.3f}   "
      f"(ecart-type {np.std(blocs_horl, ddof=1):.4f})")
print(f"  ecart par bloc       : {ecarts_bloc.min():.3f} a {ecarts_bloc.max():.3f}   "
      f"(moyenne {ecarts_bloc.mean():.4f}, ecart-type {np.std(ecarts_bloc, ddof=1):.4f})")


   T   percolation   SIR horloge   SIR par arete   ecart(perc,arete)   ecart(perc,horloge)


 0.10         0.001         0.001           0.001               0.000                0.000


 0.13         0.001         0.002           0.002               0.000                0.000


 0.16         0.005         0.005           0.006               0.000                0.000


 0.18         0.027         0.016           0.025               0.002                0.012


 0.22         0.197         0.119           0.192               0.005                0.078


 0.30         0.537         0.372           0.550               0.014                0.165

ecart maximal percolation / par arete : 0.0136
ecart maximal percolation / horloge   : 0.1645

T = 0.30 : ecart mesure sur 10 blocs de 40 tirages
  percolation par bloc : 0.530 a 0.541   (ecart-type 0.0033)
  SIR horloge par bloc : 0.309 a 0.496   (ecart-type 0.0613)
  ecart par bloc       : 0.040 a 0.227   (moyenne 0.1645, ecart-type 0.0609)


### Lecture du contrôle

Le tableau donne deux verdicts, et il faut les lire séparément.

**La file d'événements à transmission indépendante retrouve la percolation**, à `0,014` près au maximum sur toute la grille. Deux codes sans rien de commun — un tas binaire qui ordonne des dates contre un union-find qui fusionne des ensembles — mesurent la même chose. C'est la valeur du contrôle : le résultat est une propriété du graphe, non un artefact du simulateur.

**Le modèle à horloge partagée, lui, reste en dessous**, de `0,165` au plus, et ce maximum tombe à $T = 0{,}30$. L'écart n'est pas un bogue — le code est le même, seul le tirage change — et il n'est pas non plus une erreur de la théorie : la prédiction de percolation suppose des transmissions indépendantes, hypothèse que l'horloge partagée viole. Un sujet qui guérit vite perd d'un coup toutes ses tentatives ; ses arêtes échouent ensemble au lieu d'échouer séparément, et l'épidémie s'éteint plus souvent qu'une percolation de même $T$ ne le laisserait croire. L'écart se referme sous le seuil, où l'épidémie meurt de toute façon : à $T = 0{,}10$ les trois colonnes sont à `0,001`.

**La taille de cet écart n'est pas un nombre unique**, et le carnet le mesure au lieu de le supposer. Découpé en blocs de quarante tirages, à $T = 0{,}30$ où il est le plus grand, il va de `0,040` à `0,227`, de moyenne `0,164` et d'écart-type `0,061`. Cette dispersion ne vient pas de la percolation, qui tient entre `0,530` et `0,541` d'un bloc à l'autre (écart-type `0,003` sur la même moyenne) : elle est entièrement dans la colonne du SIR, qui va de `0,309` à `0,496`. C'est la signature de la bimodalité que l'exercice 1 demande de mettre au jour : un tirage isolé éteint l'épidémie ou l'embrase, et une moyenne de quarante tirages mélange les deux bosses dans des proportions qui varient d'un bloc à l'autre. Le résultat est donc « un peu plus d'un dixième », pas un chiffre au millième — et c'est la raison pour laquelle il est mesuré quatre cents fois avant d'être cité.

Deux conséquences pratiques. Un $R_0$ calculé par percolation est une **borne supérieure** de ce que donnera un SIR à durée infectieuse tirée, à $T$ égal. Et un contrôle ne juge pas une théorie : il juge un calcul. Ici il confirme les deux codes et laisse apparaître la limite du modèle — ce qui est exactement ce qu'on lui demande.

## 4. Contre-témoin — un graphe où la formule elle-même échoue

Un contrôle qui ne sait pas échouer ne prouve rien. La formule de seuil repose sur une hypothèse précise : le voisinage d'un sujet se déploie comme un arbre, sans se refermer. Quand deux voisins d'un sujet sont eux-mêmes voisins, l'infection tourne en rond au lieu de diverger.

Le graphe construit ici est le cas d'école. Chaque sommet est relié à ses deux voisins de gauche et à ses deux voisins de droite sur un anneau : il est 4-régulier, donc $\langle k\rangle = 4$ et $\langle k^2\rangle = 16$. Chaque arête appartient à un triangle, et la structure est localement unidimensionnelle.

Les deux formules annoncent une flambée : $T_c = 0{,}3333$ pour celle du réseau, $T_c = 0{,}2500$ pour le mélange homogène. La mesure, elle, dira autre chose — et à forte transmission, dira même le contraire, pour une raison de taille finie que le tableau permettra de chiffrer.

In [6]:
def anneau_triangule(n):
    """Anneau ou chaque sommet relie ses deux voisins de gauche et ses deux de droite.

    4-regulier, chaque arete dans un triangle, aucun eloignement exponentiel :
    la structure est localement unidimensionnelle."""
    adj = []
    for i in range(n):
        adj.append(sorted({(i + 1) % n, (i + 2) % n, (i - 1) % n, (i - 2) % n}))
    return adj


anneau = anneau_triangule(n_reseau)
k_moyen_a, k2_moyen_a = moment_degres(anneau)
print(f"anneau : <k>={k_moyen_a:.3f}  <k^2>={k2_moyen_a:.3f}")
print(f"formule du reseau   T_c = {k_moyen_a / (k2_moyen_a - k_moyen_a):.4f}")
print(f"melange homogene    T_c = {1.0 / k_moyen_a:.4f}")

# Temoin positif : un reseau aleatoire de meme degre moyen, meme grille de transmission.
reseau4 = construire_reseau(n_reseau, 4.0, np.random.default_rng(GRAINE + 1))
k4, k24 = moment_degres(reseau4)
print(f"reseau <k>=4 : <k^2>={k24:.3f}  T_c = {k4 / (k24 - k4):.4f}")

# Une graine par graphe : les deux colonnes ne partagent pas leur alea.
generateur_anneau = np.random.default_rng(GRAINE + 201)
generateur_temoin = np.random.default_rng(GRAINE + 202)
n_tirages_ring = 40

print("\n   T   amas(anneau)  amas(reseau)   comp(anneau)  comp(reseau)")
for T in (0.40, 0.60, 0.90):
    a = [statistiques_percolation(anneau, T, generateur_anneau) for _ in range(n_tirages_ring)]
    r = [statistiques_percolation(reseau4, T, generateur_temoin) for _ in range(n_tirages_ring)]
    amas_a = np.mean([x[0] for x in a])
    comp_a = np.mean([x[2] for x in a])
    amas_r = np.mean([x[0] for x in r])
    comp_r = np.mean([x[2] for x in r])
    print(f"{T:5.2f}   {amas_a:11.4f}   {amas_r:11.3f}   {comp_a:12.1f}   {comp_r:11.1f}")


anneau : <k>=4.000  <k^2>=16.000
formule du reseau   T_c = 0.3333
melange homogene    T_c = 0.2500
reseau <k>=4 : <k^2>=19.907  T_c = 0.2515

   T   amas(anneau)  amas(reseau)   comp(anneau)  comp(reseau)


 0.40        0.0099         0.643          853.3         765.0


 0.60        0.0297         0.880          252.8         308.4


 0.90        0.6431         0.970            3.1          86.0


### Lecture du contre-témoin

Aux transmissions modérées, le verdict est net. À $T = 0{,}40$ — au-dessus des deux seuils annoncés — l'amas le plus grand de l'anneau porte `0,99 %` de la population, contre `64,3 %` pour le réseau aléatoire de même degré moyen. À $T = 0{,}60$, l'anneau reste à `2,97 %` quand le témoin atteint `88,0 %`.

Ce n'est pas une erreur de calcul : les moments de l'anneau sont exactement ceux qu'annonce la formule. C'est l'hypothèse arborescente qui est fausse. Sur un anneau à portée deux, trois arêtes franchissent chaque point de coupe, si bien qu'un point de coupe n'est effectif que si les trois sont absentes, de probabilité $(1-T)^3$ ; le nombre attendu de coupes sur $n$ sommets vaut donc $n(1-T)^3$. Ce nombre se confronte aux composantes comptées, et l'excès est le même aux deux transmissions : `853,3` composantes contre `648` coupes attendues à $T = 0{,}40$, `252,8` contre `192` à $T = 0{,}60$ — un facteur `1,32` dans les deux cas. Les composantes sont les coupes, plus les amas isolés que la loi ne compte pas ; un excès relatif constant est ce qu'on attend de ce genre d'écart, et non un désaccord qui grandirait avec $T$.

À $T = 0{,}90$, en revanche, la mesure donne `64,3 %`, soit un très grand amas. Ce n'est pas la formule qui redevient juste : c'est la taille finie qui parle. La même loi n'annonce plus que `3` coupes attendues sur 3 000 sommets, et le tableau mesure `3,1` composantes. Un anneau coupé en trois endroits n'est plus un cercle mais trois arcs, dont le plus long porte à lui seul une large part de la population : la transition n'a pas lieu, elle est simplement hors d'atteinte d'un anneau de 3 000 sujets, où la coupure devient rare avant que la limite ne soit atteinte.

Trois conséquences pour toute lecture de $R_0$ :

- Un seuil calculé sur les degrés suppose une croissance exponentielle du voisinage. Sur une population dont les contacts se referment en cercle, cette hypothèse doit être **vérifiée**, pas supposée.
- Le contrôle de la section 3 reste valide ici : percolation et simulation s'accordent, et c'est la formule — non la mesure — qui se trompe. Un contrôle ne juge pas la théorie, il juge le calcul.
- Un désaccord entre mesure et formule n'a pas une seule cause possible : modèle faux, code faux, ou taille finie. Distinguer les trois demande une mesure supplémentaire, pas un argument.

## Exercice 1 — la taille finale est-elle bimodale au voisinage du seuil ?

Juste au-dessus du seuil, deux issues coexistent : la transmission s'éteint d'elle-même, ou elle embrase une fraction notable du réseau. Entre les deux, la distribution des tailles finales n'est pas centrée sur une valeur intermédiaire : elle a deux bosses. C'est la signature d'une transition de phase, et c'est aussi ce qui rend une moyenne trompeuse — une moyenne de quarante tirages mélange les deux bosses.

**À faire.**

1. Choisir $T$ légèrement au-dessus du seuil mesuré à la section 1.
2. Tirer 80 simulations avec ce $T$, graines distinctes, et relever la taille finale de chacune.
3. Regrouper les tailles en classes (0–1 %, 1–5 %, 5–20 %, au-delà) et compter les effectifs.
4. Conclure : deux bosses ou une seule ? Et que vaut la moyenne, comparée à chacune des deux bosses ?

In [7]:
T_exercice1 = None        # TODO etudiant : une valeur juste au-dessus du seuil mesure
tailles_finales = None    # TODO etudiant : 80 appels a epidemie(...), graines distinctes

# Etape 1 : T se choisit a partir du seuil imprime en section 1.
# Etape 2 : compter les effectifs par classe de taille, puis afficher le tableau.
# Etape 3 : comparer la moyenne des 80 tirages a la position des deux bosses.

print("Exercice 1 a completer : distribution des tailles finales au voisinage du seuil")

Exercice 1 a completer : distribution des tailles finales au voisinage du seuil


## Exercice 2 — où se situe le seuil, mesuré par dichotomie ?

Les sections précédentes ont annoncé un seuil et l'ont illustré par une grille de valeurs. L'exercice le mesure, et compare trois estimations.

**À faire.**

1. Construire deux réseaux à degré moyen 3 et 10.
2. Pour chacun, calculer les deux seuils théoriques à partir des moments mesurés.
3. Mesurer la transition par percolation statique : chercher par dichotomie la valeur de $T$ où la fraction du plus grand amas franchit la moitié de la population.
4. Comparer les trois nombres, et dire lequel des deux seuils théoriques la mesure suit.

In [8]:
seuils_exercice2 = None   # TODO etudiant : dict {degre_moyen: (seuil_reseau, seuil_homogene, seuil_mesure)}

# Etape 1 : construire_reseau(n_reseau, 3.0, ...) et construire_reseau(n_reseau, 10.0, ...).
# Etape 2 : moment_degres(...) puis les deux formules.
# Etape 3 : dichotomie sur statistiques_percolation(...)[0] jusqu'a un ecart de 0.005 en T.
# Etape 4 : afficher les trois valeurs cote a cote et commenter.

print("Exercice 2 a completer : seuils pour <k> = 3 et <k> = 10")

Exercice 2 a completer : seuils pour <k> = 3 et <k> = 10


## Exercice 3 — combien d'arêtes lointaines faut-il pour réparer l'anneau ?

Le contre-témoin a montré qu'un anneau ne flambe pas, faute d'expansion. La question naturelle est quantitative : combien d'arêtes tirées au hasard faut-il ajouter pour que le graphe retrouve un amas géant, et les formules redeviennent-elles alors prédictives ?

**À faire.**

1. Ajouter à l'anneau triangulé un nombre $q$ d'arêtes aléatoires entre sommets quelconques, pour $q$ croissant (1, 2, 4, 8, 16, 32).
2. Pour chaque $q$, mesurer la fraction de l'amas géant à $T = 0{,}60$.
3. Recalculer les moments de degrés du graphe obtenu et son seuil de réseau.
4. Dire à partir de quel $q$ la mesure cesse de contredire la formule, et pourquoi ce $q$ dépend de la taille du réseau. Le lien avec la loi $n(1-T)^3$ du contre-témoin est la piste à creuser.

In [9]:
q_critique = None        # TODO etudiant : premier q ou la mesure suit la formule

# Etape 1 : copier l'anneau en listes modifiables, puis ajouter q aretes aleatoires.
# Etape 2 : mesurer l'amas geant a T = 0.60 pour chaque q.
# Etape 3 : recalculer moment_degres(...) et le seuil du graphe modifie.
# Etape 4 : commenter le lien entre q, la taille n et l'hypothese arborescente.

print("Exercice 3 a completer : reparation de l'anneau par aretes lointaines")

Exercice 3 a completer : reparation de l'anneau par aretes lointaines


## 5. Statut de ce que le carnet établit

**Mesuré dans ce carnet.**

- Les deux premiers moments de degrés du réseau de référence, `<k>` = `6,000` et `<k^2>` = `42,124`, et les deux seuils qui s'en déduisent, $0{,}1661$ pour la formule de réseau et $0{,}1667$ pour le mélange homogène.
- L'accord entre la percolation statique et la file d'événements à transmission indépendante, à `0,014` près au maximum sur six valeurs de $T$ : deux codes sans structures communes rendent le même nombre.
- Le désaccord, mesuré et attribué, entre percolation et SIR à horloge partagée : `0,165` au maximum. Le même simulateur, avec des transmissions indépendantes, referme l'écart — c'est la corrélation entre les arêtes d'un même sujet, non le code, qui produit la différence. Le chiffre porte sa dispersion : sur des blocs de quarante tirages, l'écart va de `0,040` à `0,227` (écart-type `0,061`), parce que la taille finale est elle-même bimodale.
- Le contre-témoin : sur l'anneau triangulé, à $T = 0{,}60$, l'amas le plus grand porte `2,97 %` de la population contre `88,0 %` pour un réseau aléatoire de même degré moyen ; à $T = 0{,}90$, la même mesure donne `64,3 %`, et le nombre de composantes `3,1` confronte la loi $n(1-T)^3$ à la taille finie.

**Cité, et non vérifié ici.**

Trois énoncés classiques sont invoqués sans démonstration : le seuil $\langle k\rangle/(\langle k^2\rangle-\langle k\rangle)$ pour un réseau à degrés prescrits, l'équivalence entre transmissions indépendantes et amas de percolation, et la loi des coupes d'un anneau à portée deux. Aucune publication n'a été dépouillée pour écrire ce carnet ; ces énoncés servent d'hypothèses de travail, et c'est la mesure qui les éprouve — y compris en les prenant en défaut, section 4.

**Ouvert.**

- La validité sur des populations réelles : un réseau de contacts observé n'est ni poissonnien, ni sans triangles, ni fixe sur la durée d'une épidémie.
- L'inférence : retrouver $T$ et la structure à partir d'une courbe épidémique observée est un problème inverse, non traité.
- Le raccord quantitatif entre les deux modèles de transmission : le carnet mesure l'écart et sa dispersion, il ne les calcule pas. Une expression de la taille finale pour une horloge partagée reste à établir.

### Limites

Le modèle suppose une population fermée, un graphe fixe, une transmission homogène et une immunité définitive. Il ignore la période latente, la structure d'âge, la saisonnalité et les comportements adaptatifs — ce qui, dans une épidémie réelle, décide souvent de la suite. Une taille finale mesurée ici n'est pas une prédiction : c'est la conséquence exacte d'hypothèses choisies.

## Pour aller plus loin

- **La feuille précédente** — [Dynamiques-01](Dynamiques-01-Populations-et-Epidemies-Python.ipynb) — traite le SIR à population homogène et son équation de taille finale, résolue sans intégrer le système. Le présent carnet est son prolongement : mêmes compartiments, mais des contacts qui ont une forme.
- **Gillespie et le passage au déterministe** — la même file d'événements, appliquée à une population mélangée, doit retrouver l'équation différentielle quand le volume croît. C'est l'extension suivante de la sous-série.
- **Inférence des paramètres** — [Probas/PyMC](../../Probas/PyMC/README.md) porte l'inférence bayésienne qui permettrait de remonter de la courbe observée aux paramètres.
- **Théorie des graphes** — [Search](../../Search/README.md) et [GameTheory](../../GameTheory/README.md) donnent les outils de parcours et de dynamique sur graphe qui sous-tendent le union-find et la percolation employés ici.

[↑ Dynamiques](README.md) · [↑ Bio & Médecine](../README.md)